# Kairos-CIDA-7B · Notebook 1: World-Class Data Preparation Pipeline

Build a clean, deduplicated, multi-task reasoning dataset (~200 k examples)
blending **Math** (GSM8K + MATH), **Code** (MBPP + CodeAlpaca), and
**Logic / Reasoning** (ARC-Challenge + ARC-Easy + Big-Bench Hard).

All examples are formatted with a uniform ChatML template, tokenized with
left-padded prompts and right-padded answers, and saved as `.arrow` shards
ready for the training notebook.

**Runtime:** CPU or T4 GPU. No model weights are loaded.

## 1 · Install Dependencies

In [ ]:
!pip install -q transformers datasets datasketch tqdm
print("OK")

## 2 · Imports & Configuration

In [ ]:
import re, os, json, hashlib, unicodedata, warnings
from pathlib import Path
from typing import Optional, List, Dict
from dataclasses import dataclass, asdict

import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, Dataset as HFDataset, DatasetDict
from transformers import AutoTokenizer
from datasketch import MinHash, MinHashLSH
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

@dataclass
class DataConfig:
    model_name:        str   = "Qwen/Qwen1.5-7B-Chat"
    fallback_model:    str   = "Qwen/Qwen1.5-1.8B-Chat"
    max_prompt_tokens: int   = 384
    max_answer_tokens: int   = 128
    gsm8k_limit:       int   = 8000
    math_limit:        int   = 30000
    mbpp_limit:        int   = 4000
    codealpaca_limit:  int   = 20000
    arc_limit:         int   = 14000
    bbh_limit:         int   = 8000
    total_cap:         int   = 200_000
    minhash_threshold: float = 0.85
    minhash_num_perm:  int   = 128
    output_dir:        str   = "kairos_dataset"
    val_fraction:      float = 0.02

cfg = DataConfig()
print("Config OK:", asdict(cfg))


## 3 · Load Tokenizer (for Length Filtering)

In [ ]:
try:
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
    print(f"Loaded {cfg.model_name}")
except Exception as e:
    print(f"Falling back: {e}")
    tokenizer = AutoTokenizer.from_pretrained(cfg.fallback_model, trust_remote_code=True)
    print(f"Loaded {cfg.fallback_model}")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"vocab_size={tokenizer.vocab_size}, pad_token={repr(tokenizer.pad_token)}")


## 4 · Text Cleaning & Sanitization Utilities

In [ ]:
_REPLACEMENT_CHAR = "\ufffd"
_CONTROL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

def is_clean_text(text: str) -> bool:
    """Return False if the text has encoding artifacts or control chars."""
    if not isinstance(text, str) or not text.strip():
        return False
    if _REPLACEMENT_CHAR in text:
        return False
    if _CONTROL_RE.search(text):
        return False
    try:
        unicodedata.normalize("NFC", text)
    except Exception:
        return False
    return True

def extract_gsm8k_answer(answer_text: str) -> Optional[str]:
    """Extract numeric answer after '####' in GSM8K."""
    m = re.search(r"####\s*([\-0-9,\.]+)", answer_text)
    return m.group(1).replace(",", "").strip() if m else None

def extract_math_boxed(solution: str) -> Optional[str]:
    """Extract content inside \\boxed{...} — handles nested braces."""
    idx = solution.rfind(r"\boxed{")
    if idx == -1:
        return None
    depth, start = 0, idx + len(r"\boxed{") - 1
    for i in range(start, len(solution)):
        if solution[i] == "{":
            depth += 1
        elif solution[i] == "}":
            depth -= 1
            if depth == 0:
                return solution[start + 1 : i].strip()
    return None

def extract_code_block(text: str) -> Optional[str]:
    """Extract first ```python...``` block."""
    m = re.search(r"```python\s*(.*?)```", text, re.DOTALL)
    if m:
        code = m.group(1).strip()
        return code if len(code) > 20 else None
    return None

def is_valid_python(code: str) -> bool:
    try:
        compile(code, "<string>", "exec")
        return True
    except SyntaxError:
        return False

_tok_cache: Dict[str, int] = {}
def token_len(text: str) -> int:
    if text not in _tok_cache:
        _tok_cache[text] = len(tokenizer.encode(text, add_special_tokens=False))
    return _tok_cache[text]

def passes_length_filter(prompt: str, answer: str) -> bool:
    return (token_len(prompt) <= cfg.max_prompt_tokens
            and token_len(answer) <= cfg.max_answer_tokens)

print("Cleaning utilities OK")


## 5 · Unified ChatML Template

In [ ]:
SYSTEM_PROMPTS = {
    "math": (
        "You are a precise mathematical reasoner. "
        "Solve the problem step by step and end with '#### <numeric answer>'."
    ),
    "code": (
        "You are an expert Python programmer. "
        "Write clean, correct, well-commented code inside a ```python block."
    ),
    "logic": (
        "You are a logical reasoning expert. "
        "Analyse the problem carefully and select the best answer option."
    ),
}

def make_chatml_prompt(question: str, task: str) -> str:
    sys_prompt = SYSTEM_PROMPTS.get(task, SYSTEM_PROMPTS["logic"])
    return (
        f"<|im_start|>system\n{sys_prompt}\n<|im_end|>\n"
        f"<|im_start|>user\n{question.strip()}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

sample = make_chatml_prompt("What is 2+2?", "math")
print("Sample prompt (repr):\n", repr(sample))
print(f"Token length: {token_len(sample)}")


## 6 · MinHash Deduplication Engine

In [ ]:
class MinHashDeduplicator:
    """
    Identifies near-duplicate questions using MinHash LSH.
    Two questions are duplicates if their 5-gram Jaccard similarity >= threshold.
    First occurrence is kept; subsequent near-duplicates are discarded.
    """
    def __init__(self, threshold: float = 0.85, num_perm: int = 128):
        self.lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
        self.num_perm = num_perm
        self.seen_exact: set = set()
        self.n_exact = 0
        self.n_near  = 0
        self._ctr    = 0

    def _shingles(self, text: str) -> set:
        t = re.sub(r"\s+", " ", text.lower().strip())
        return {t[i:i+5] for i in range(len(t) - 4)}

    def _minhash(self, text: str) -> MinHash:
        mh = MinHash(num_perm=self.num_perm)
        for s in self._shingles(text):
            mh.update(s.encode("utf8"))
        return mh

    def is_duplicate(self, text: str) -> bool:
        h = hashlib.md5(text.encode()).hexdigest()
        if h in self.seen_exact:
            self.n_exact += 1
            return True
        self.seen_exact.add(h)
        mh = self._minhash(text)
        key = str(self._ctr); self._ctr += 1
        if self.lsh.query(mh):
            self.n_near += 1
            return True
        try:
            self.lsh.insert(key, mh)
        except ValueError:
            pass
        return False

    def stats(self) -> dict:
        return {"exact": self.n_exact, "near": self.n_near,
                "total": self.n_exact + self.n_near}

deduplicator = MinHashDeduplicator(cfg.minhash_threshold, cfg.minhash_num_perm)
print(f"MinHash deduplicator ready  threshold={cfg.minhash_threshold}")


## 7 · Math Dataset Loaders (GSM8K + MATH)

In [ ]:
def load_gsm8k(limit: int) -> List[dict]:
    raw = load_dataset("openai/gsm8k", "main", split="train")
    records, skipped = [], 0
    for ex in tqdm(raw, desc="GSM8K", total=len(raw)):
        if len(records) >= limit:
            break
        q   = ex["question"].strip()
        ans = extract_gsm8k_answer(ex["answer"])
        if ans is None or not (is_clean_text(q) and is_clean_text(ans)):
            skipped += 1; continue
        full_answer = ex["answer"].strip()
        prompt = make_chatml_prompt(q, "math")
        if not passes_length_filter(prompt, full_answer):
            skipped += 1; continue
        if deduplicator.is_duplicate(q):
            skipped += 1; continue
        records.append({"prompt": prompt, "answer": full_answer,
                        "task": "math", "source": "gsm8k"})
    print(f"  GSM8K      -> {len(records):>6} kept | {skipped} skipped")
    return records

def load_math(limit: int) -> List[dict]:
    try:
        raw = load_dataset("lighteval/MATH", "all", split="train", trust_remote_code=True)
    except Exception:
        try:
            raw = load_dataset("hendrycks/competition_math", split="train", trust_remote_code=True)
        except Exception:
            print("  MATH dataset unavailable; skipping.")
            return []

    KEEP_SUBJECTS = {"algebra", "counting_and_probability", "number_theory",
                     "precalculus", "prealgebra"}
    records, skipped = [], 0
    for ex in tqdm(raw, desc="MATH", total=min(len(raw), limit * 3)):
        if len(records) >= limit:
            break
        problem  = (ex.get("problem") or ex.get("question") or "").strip()
        solution = (ex.get("solution") or ex.get("answer") or "").strip()
        subject  = (ex.get("type") or ex.get("subject") or "").lower().replace(" ", "_")
        level    = str(ex.get("level") or "")
        lvl_m = re.search(r"(\d)", level)
        if lvl_m and int(lvl_m.group(1)) > 3:
            skipped += 1; continue
        if subject and subject not in KEEP_SUBJECTS:
            skipped += 1; continue
        if extract_math_boxed(solution) is None:
            skipped += 1; continue
        if not (is_clean_text(problem) and is_clean_text(solution)):
            skipped += 1; continue
        prompt = make_chatml_prompt(problem, "math")
        if not passes_length_filter(prompt, solution):
            skipped += 1; continue
        if deduplicator.is_duplicate(problem):
            skipped += 1; continue
        records.append({"prompt": prompt, "answer": solution,
                        "task": "math", "source": "math_competition"})
    print(f"  MATH       -> {len(records):>6} kept | {skipped} skipped")
    return records

print("Math loaders defined")


## 8 · Code & Logic Dataset Loaders

In [ ]:
def load_mbpp(limit: int) -> List[dict]:
    try:
        raw = load_dataset("google-research-datasets/mbpp", "sanitized", split="train", trust_remote_code=True)
    except Exception:
        raw = load_dataset("google-research-datasets/mbpp", split="train", trust_remote_code=True)
    records, skipped = [], 0
    for ex in tqdm(raw, desc="MBPP", total=min(len(raw), limit)):
        if len(records) >= limit:
            break
        question = (ex.get("text") or ex.get("prompt") or "").strip()
        code     = (ex.get("code") or "").strip()
        if not (is_clean_text(question) and is_clean_text(code)):
            skipped += 1; continue
        if not is_valid_python(code):
            skipped += 1; continue
        answer = f"```python\n{code}\n```"
        prompt = make_chatml_prompt(question, "code")
        if not passes_length_filter(prompt, answer):
            skipped += 1; continue
        if deduplicator.is_duplicate(question):
            skipped += 1; continue
        records.append({"prompt": prompt, "answer": answer,
                        "task": "code", "source": "mbpp"})
    print(f"  MBPP       -> {len(records):>6} kept | {skipped} skipped")
    return records

def load_codealpaca(limit: int) -> List[dict]:
    try:
        raw = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
    except Exception:
        print("  CodeAlpaca unavailable; skipping.")
        return []
    records, skipped = [], 0
    for ex in tqdm(raw, desc="CodeAlpaca", total=min(len(raw), limit * 2)):
        if len(records) >= limit:
            break
        instruction = (ex.get("instruction") or "").strip()
        output      = (ex.get("output") or "").strip()
        if not (is_clean_text(instruction) and is_clean_text(output)):
            skipped += 1; continue
        if "```python" not in output and "def " not in output:
            skipped += 1; continue
        code = extract_code_block(output)
        if code and not is_valid_python(code):
            skipped += 1; continue
        prompt = make_chatml_prompt(instruction, "code")
        if not passes_length_filter(prompt, output):
            skipped += 1; continue
        if deduplicator.is_duplicate(instruction):
            skipped += 1; continue
        records.append({"prompt": prompt, "answer": output,
                        "task": "code", "source": "codealpaca"})
    print(f"  CodeAlpaca -> {len(records):>6} kept | {skipped} skipped")
    return records

def load_arc(limit: int) -> List[dict]:
    records, skipped = [], 0
    for config_name, label in [("ARC-Challenge", "arc_challenge"),
                                ("ARC-Easy",      "arc_easy")]:
        if len(records) >= limit:
            break
        raw = load_dataset("allenai/ai2_arc", config_name, split="train")
        per_split_limit = limit // 2
        for ex in tqdm(raw, desc=config_name, total=min(len(raw), per_split_limit)):
            if len(records) >= limit:
                break
            question = ex["question"].strip()
            choices  = ex["choices"]
            ans_key  = ex["answerKey"].strip().upper()
            if not ans_key or not is_clean_text(question):
                skipped += 1; continue
            choice_text = "\n".join(
                f"{lbl}. {txt}"
                for lbl, txt in zip(choices["label"], choices["text"])
            )
            full_q = f"{question}\n{choice_text}"
            try:
                idx    = choices["label"].index(ans_key)
                answer = f"{ans_key}. {choices['text'][idx]}"
            except (ValueError, IndexError):
                skipped += 1; continue
            prompt = make_chatml_prompt(full_q, "logic")
            if not passes_length_filter(prompt, answer):
                skipped += 1; continue
            if deduplicator.is_duplicate(question):
                skipped += 1; continue
            records.append({"prompt": prompt, "answer": answer,
                            "task": "logic", "source": label})
    print(f"  ARC        -> {len(records):>6} kept | {skipped} skipped")
    return records

def load_bbh(limit: int) -> List[dict]:
    BBH_TASKS = [
        "logical_deduction_three_objects", "logical_deduction_five_objects",
        "logical_deduction_seven_objects", "causal_judgement",
        "formal_fallacies", "navigate",
        "reasoning_about_colored_objects",
        "tracking_shuffled_objects_three_objects",
        "word_sorting", "boolean_expressions",
    ]
    records, skipped = [], 0
    per_task = max(1, limit // len(BBH_TASKS))
    for task in BBH_TASKS:
        if len(records) >= limit:
            break
        try:
            raw = load_dataset("lukaemon/bbh", task, split="test", trust_remote_code=True)
        except Exception:
            continue
        for ex in tqdm(raw, desc=f"BBH/{task}", total=min(len(raw), per_task)):
            if len(records) >= limit:
                break
            question = (ex.get("input") or ex.get("question") or "").strip()
            answer   = (ex.get("target") or ex.get("answer") or "").strip()
            if not (is_clean_text(question) and is_clean_text(answer)):
                skipped += 1; continue
            prompt = make_chatml_prompt(question, "logic")
            if not passes_length_filter(prompt, answer):
                skipped += 1; continue
            if deduplicator.is_duplicate(question):
                skipped += 1; continue
            records.append({"prompt": prompt, "answer": answer,
                            "task": "logic", "source": f"bbh_{task}"})
    print(f"  BBH        -> {len(records):>6} kept | {skipped} skipped")
    return records

print("Code + Logic loaders defined")


## 9 · Execute All Loaders & Blend

In [ ]:
import random
random.seed(42)

print("=" * 60)
print("Loading and cleaning all datasets...")
print("=" * 60)

all_records: List[dict] = []
all_records.extend(load_gsm8k(cfg.gsm8k_limit))
all_records.extend(load_math(cfg.math_limit))
all_records.extend(load_mbpp(cfg.mbpp_limit))
all_records.extend(load_codealpaca(cfg.codealpaca_limit))
all_records.extend(load_arc(cfg.arc_limit))
all_records.extend(load_bbh(cfg.bbh_limit))

random.shuffle(all_records)
all_records = all_records[:cfg.total_cap]

task_counts: Dict[str, int] = {}
for r in all_records:
    task_counts[r["task"]] = task_counts.get(r["task"], 0) + 1

print(f"\nTotal: {len(all_records):,}")
for t, c in sorted(task_counts.items()):
    print(f"  {t}: {c:,}  ({100*c/len(all_records):.1f}%)")

dup_stats = deduplicator.stats()
print(f"Dups removed: {dup_stats}")


## 10 · Tokenize & Create PyTorch-Ready Dataset

**Padding strategy:**
- Prompt: **left-pad** to `MAX_PROMPT_TOKENS` (required for causal LM generation)
- Answer: **right-pad** to `MAX_ANSWER_TOKENS`

The training notebook will concatenate `[prompt_embeds | plan_embeds | answer_embeds]`
in embedding space, so each piece needs a fixed length for clean batching.

In [ ]:
MAX_PROMPT_TOKENS = cfg.max_prompt_tokens   # 384
MAX_ANSWER_TOKENS = cfg.max_answer_tokens   # 128

class KairosDataset(Dataset):
    """
    Pre-tokenized dataset for Kairos-CIDA training.

    Returned tensors per example
    ----------------------------
    prompt_ids   (MAX_PROMPT_TOKENS,)  int64  left-padded prompt
    prompt_mask  (MAX_PROMPT_TOKENS,)  int64  1=real token, 0=pad
    answer_ids   (MAX_ANSWER_TOKENS,)  int64  right-padded gold answer
    answer_mask  (MAX_ANSWER_TOKENS,)  int64  1=real token, 0=pad
    task         str                          "math"|"code"|"logic"
    """
    def __init__(self, records: List[dict]):
        self.records = records
        self.pad_id  = tokenizer.pad_token_id

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int) -> dict:
        rec = self.records[idx]

        # Prompt: left-pad (critical for causal LM — last real token must be rightmost)
        tokenizer.padding_side = "left"
        p_enc = tokenizer(
            rec["prompt"],
            max_length=MAX_PROMPT_TOKENS, truncation=True,
            padding="max_length", return_tensors="pt",
        )

        # Answer: right-pad (loss mask will cover padding)
        tokenizer.padding_side = "right"
        a_enc = tokenizer(
            rec["answer"],
            max_length=MAX_ANSWER_TOKENS, truncation=True,
            padding="max_length", return_tensors="pt",
        )
        tokenizer.padding_side = "left"  # restore default

        return {
            "prompt_ids":  p_enc["input_ids"].squeeze(0),
            "prompt_mask": p_enc["attention_mask"].squeeze(0),
            "answer_ids":  a_enc["input_ids"].squeeze(0),
            "answer_mask": a_enc["attention_mask"].squeeze(0),
            "task":        rec["task"],
        }

print("KairosDataset defined")
print(f"  Prompt: {MAX_PROMPT_TOKENS} tokens (left-padded)")
print(f"  Answer: {MAX_ANSWER_TOKENS} tokens (right-padded)")


## 11 · Train/Val Split, Sanity Checks & Save

In [ ]:
random.seed(42)
n_val   = max(200, int(len(all_records) * cfg.val_fraction))
n_train = len(all_records) - n_val
train_records = all_records[:n_train]
val_records   = all_records[n_train:]

train_ds = KairosDataset(train_records)
val_ds   = KairosDataset(val_records)
print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")

# Sanity check
sample = train_ds[0]
pad_id = tokenizer.pad_token_id
print(f"\nSample 0:")
print(f"  prompt_ids  shape  : {sample['prompt_ids'].shape}")
print(f"  answer_ids  shape  : {sample['answer_ids'].shape}")
print(f"  prompt left-pad ?  : {(sample['prompt_ids'][:5] == pad_id).all().item()}  "
      f"(first 5 ids: {sample['prompt_ids'][:5].tolist()})")
print(f"  answer right-pad ? : {(sample['answer_ids'][-3:] == pad_id).any().item()}  "
      f"(last 3 ids: {sample['answer_ids'][-3:].tolist()})")
real_prompt_len = sample['prompt_mask'].sum().item()
real_answer_len = sample['answer_mask'].sum().item()
print(f"  real prompt tokens : {real_prompt_len}")
print(f"  real answer tokens : {real_answer_len}")

# Attention mask assertions
assert (sample['prompt_ids'][:int(sample['prompt_ids'].size(0) - real_prompt_len)] == pad_id).all(), \
    "Left-padding violated on prompt!"
assert sample['prompt_mask'][-1].item() == 1, "Last prompt token must not be pad!"

# DataLoader batch test
loader = DataLoader(train_ds, batch_size=4, shuffle=False)
batch  = next(iter(loader))
print(f"\nBatch shapes: prompt={batch['prompt_ids'].shape}, answer={batch['answer_ids'].shape}")
print(f"Real prompt tokens per sample: {batch['prompt_mask'].sum(-1).tolist()}")
print(f"Real answer tokens per sample: {batch['answer_mask'].sum(-1).tolist()}")
print("All sanity checks PASSED")


In [ ]:
out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

def save_jsonl(records: List[dict], path: Path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

save_jsonl(train_records, out_dir / "train.jsonl")
save_jsonl(val_records,   out_dir / "val.jsonl")

meta = {
    "model_name":        cfg.model_name,
    "max_prompt_tokens": MAX_PROMPT_TOKENS,
    "max_answer_tokens": MAX_ANSWER_TOKENS,
    "n_train":           len(train_records),
    "n_val":             len(val_records),
    "task_distribution": task_counts,
    "dedup_stats":       dup_stats,
    "pad_token_id":      tokenizer.pad_token_id,
    "eos_token_id":      tokenizer.eos_token_id,
}
with open(out_dir / "dataset_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

# HuggingFace Arrow format for fast random-access in training
HFDataset.from_list(train_records).save_to_disk(str(out_dir / "arrow" / "train"))
HFDataset.from_list(val_records).save_to_disk(str(out_dir / "arrow" / "val"))

print(f"\nSaved to '{out_dir}/':")
for p in sorted(out_dir.rglob("*")):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  {p.relative_to(out_dir)}  ({size_kb:.0f} KB)")
print("Dataset ready for Notebook 2.")


## 12 · Dataset Quality Report

In [ ]:
print("=" * 65)
print("  KAIROS DATASET — QUALITY REPORT")
print("=" * 65)
print(f"  Total examples : {len(all_records):>10,}")
print(f"  Train          : {len(train_records):>10,}")
print(f"  Validation     : {len(val_records):>10,}")
print()
print("  Task distribution:")
for task, cnt in sorted(task_counts.items()):
    pct = 100 * cnt / len(all_records)
    bar = "█" * int(pct / 2)
    print(f"    {task:<8} {cnt:>8,}  ({pct:5.1f}%)  {bar}")
print()
print("  Source breakdown:")
source_counts: Dict[str, int] = {}
for r in all_records:
    source_counts[r["source"]] = source_counts.get(r["source"], 0) + 1
for src, cnt in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"    {src:<36} {cnt:>8,}")
print()
print("  Deduplication:")
print(f"    Exact dups removed : {dup_stats['exact']:>8,}")
print(f"    Near  dups removed : {dup_stats['near']:>8,}")
print()
print(f"  Max prompt tokens  : {MAX_PROMPT_TOKENS}")
print(f"  Max answer tokens  : {MAX_ANSWER_TOKENS}")
print("=" * 65)
print("  Dataset ready for Notebook 2 (Training Pipeline)")
print("=" * 65)
